<a href="https://colab.research.google.com/github/Chaos-19/practical_deep_learning_for_coders/blob/main/movie_genra_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
from pathlib import Path
from tqdm import tqdm

# ==========================================
# CONFIG
# ==========================================

TMDB_API_KEY = "ec0f61a94f57e512d9d9d06d8ad99a5e"

DATASET_DIR = Path("movie_posters")

GENRES = {
    "action": 28,
    "comedy": 35,
    "horror": 27,
    "romance": 10749,
}

MOVIES_PER_GENRE = 280

# ==========================================
# HELPERS
# ==========================================

def safe_filename(name):
    return "".join(c for c in name if c.isalnum() or c in (" ", "_", "-")).rstrip()

def get_movies_by_genre(genre_id, pages=5):
    movies = []

    for page in range(1, pages + 1):
        url = "https://api.themoviedb.org/3/discover/movie"

        params = {
            "api_key": TMDB_API_KEY,
            "with_genres": genre_id,
            "sort_by": "popularity.desc",
            "page": page,
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()
        movies.extend(data["results"])

    return movies


def download_poster(movie, save_dir):
    poster_path = movie.get("poster_path")

    if not poster_path:
        return False

    poster_url = f"https://image.tmdb.org/t/p/w500{poster_path}"

    title = safe_filename(movie["title"])
    filepath = save_dir / f"{title}.jpg"

    if filepath.exists():
        return True

    try:
        img = requests.get(poster_url, timeout=10)

        if img.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(img.content)
            return True

    except Exception:
        pass

    return False


# ==========================================
# DOWNLOAD DATASET
# ==========================================

DATASET_DIR.mkdir(exist_ok=True)

for genre_name, genre_id in GENRES.items():

    genre_dir = DATASET_DIR / genre_name
    genre_dir.mkdir(exist_ok=True)

    print(f"\nDownloading {genre_name} posters...")

    movies = get_movies_by_genre(genre_id)

    count = 0

    for movie in tqdm(movies):

        if count >= MOVIES_PER_GENRE:
            break

        success = download_poster(movie, genre_dir)

        if success:
            count += 1

    print(f"Saved {count} posters to {genre_dir}")

print("\nDataset ready!")

In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *

In [ ]:
#id first_training
#caption Results from the first training
# CLICK ME
from fastai.vision.all import *
path = Path('/content/movie_posters')

dls = ImageDataLoaders.from_folder(
    path,
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(460),
    batch_tfms=aug_transforms(size=224)
)

learn = vision_learner(
    dls,
    resnet34,
    metrics=accuracy
)

learn.fine_tune(3)

In [ ]:
dls.show_batch(max_n=9)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)

interp.plot_confusion_matrix(figsize=(8,8))

In [ ]:
interp.plot_top_losses(9)

In [ ]:
from collections import Counter

labels = [p.parent.name for p in get_image_files(path)]
Counter(labels)

In [ ]:
learn.show_results(max_n=16)

In [ ]:
from fastai.vision.all import *
from fastai.vision.widgets import *

cleaner = ImageClassifierCleaner(learn)
cleaner

In [ ]:
# for idx in cleaner.delete(): cleaner.fns[idx].unlink()
# for idx,movie in cleaner.change(): shutil.move(str(cleaner.fns[idx]), path/movie_posters)

In [ ]:
learn.export('model.pkl')